In [1]:
from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_classic.retrievers import MultiQueryRetriever
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [2]:
load_dotenv()

True

In [3]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

llm = ChatOpenAI(model="gpt-5-mini", temperature=0.3)

In [4]:
docs = [
    Document(
        page_content=(
            "Biotechnology companies are developing novel protein-based therapies that target specific "
            "disease pathways with unprecedented precision. Synthetic biology techniques allow scientists "
            "to engineer microorganisms that produce pharmaceutical compounds at industrial scale. "
            "Bioreactor technologies have dramatically reduced the cost of producing monoclonal antibodies, "
            "making treatments for autoimmune diseases and cancers more accessible. Microbiome research is "
            "revealing how manipulating gut bacteria can influence everything from mental health to "
            "metabolic disorders."
        ),
        metadata={"topic": "biotechnology"},
    ),
    Document(
        page_content=(
            "Zero-trust architecture has become the gold standard for enterprise network security, "
            "requiring continuous verification rather than relying on perimeter defenses. Machine learning "
            "models now detect anomalous network behavior in real time, reducing the window between "
            "intrusion and detection from months to minutes. Ransomware attacks on critical infrastructure "
            "have forced governments to establish mandatory incident reporting requirements for healthcare "
            "and energy sectors. Post-quantum cryptography standards are being finalized to protect "
            "sensitive data against future quantum computing threats."
        ),
        metadata={"topic": "cybersecurity"},
    ),
    Document(
        page_content=(
            "Brain-computer interfaces are enabling paralyzed patients to control prosthetic limbs and "
            "communicate using only their neural signals. Optogenetics allows researchers to activate or "
            "silence specific neuron populations with light, accelerating the understanding of neural "
            "circuit function and disease. Advanced neuroimaging techniques using fMRI and "
            "magnetoencephalography are mapping brain connectivity with millimeter precision, unlocking "
            "new treatments for depression and PTSD. Neurofeedback therapies are showing promise for "
            "cognitive rehabilitation following traumatic brain injuries."
        ),
        metadata={"topic": "neuroscience"},
    ),
    Document(
        page_content=(
            "Perovskite solar cells have achieved efficiency ratings exceeding 33%, surpassing traditional "
            "silicon panels and promising dramatically lower manufacturing costs. Grid-scale battery "
            "storage using iron-air and sodium-ion technologies is making renewable energy dispatchable "
            "around the clock without relying on rare earth metals. Offshore floating wind farms are "
            "expanding into deep-water regions previously inaccessible to fixed-foundation turbines, "
            "multiplying available wind energy capacity. Green hydrogen produced via electrolysis is "
            "emerging as a critical energy carrier for decarbonizing heavy industry and long-haul "
            "transport."
        ),
        metadata={"topic": "renewable_energy"},
    ),
    Document(
        page_content=(
            "Surgical robots equipped with haptic feedback allow surgeons to perform minimally invasive "
            "procedures with sub-millimeter precision, reducing patient recovery times significantly. "
            "Collaborative robots in manufacturing now work safely alongside humans using advanced "
            "computer vision and force sensing, without the need for physical barriers. Autonomous mobile "
            "robots are transforming warehouse logistics, optimizing pick-and-place operations and "
            "reducing fulfillment errors. Soft robots inspired by biological organisms are being developed "
            "for delicate tasks in agriculture, search-and-rescue, and medical drug delivery."
        ),
        metadata={"topic": "robotics"},
    ),
    Document(
        page_content=(
            "Base editing and prime editing technologies offer more precise alternatives to CRISPR-Cas9, "
            "enabling single-letter corrections to the genome without creating double-strand breaks. "
            "Gene therapy trials using adeno-associated virus vectors have achieved functional cures for "
            "hemophilia B and spinal muscular atrophy. Epigenome editing tools allow researchers to "
            "switch genes on or off without altering the underlying DNA sequence, opening new avenues "
            "for treating complex diseases. Polygenic risk scoring combined with germline analysis is "
            "enabling predictive medicine that identifies disease susceptibility decades before symptoms "
            "appear."
        ),
        metadata={"topic": "genetic_engineering"},
    ),
]

print(f"Created {len(docs)} documents")

Created 6 documents


In [5]:
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)

docs = splitter.split_documents(docs)

print(f"Chunks: {len(docs)}")

Chunks: 18


### input_query --> llm --> 3 queries (variants) --> retrieval --> 9 docs --> deduplication

In [6]:
vectorstore = InMemoryVectorStore.from_documents(docs, embedding=embeddings)

base_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [7]:
results_with_sim_search = base_retriever.invoke("How are modern technologies improving human health?")

print(f"Retrieved {len(results_with_sim_search)} unique documents:\n")
for i, doc in enumerate(results_with_sim_search):
    print(f"--- Result {i+1} [{doc.metadata.get('topic')}] ---")
    print(doc.page_content)
    print()

Retrieved 3 unique documents:

--- Result 1 [biotechnology] ---
Biotechnology companies are developing novel protein-based therapies that target specific disease pathways with unprecedented precision. Synthetic biology techniques allow scientists to engineer microorganisms that produce pharmaceutical compounds at industrial scale. Bioreactor technologies have

--- Result 2 [robotics] ---
Surgical robots equipped with haptic feedback allow surgeons to perform minimally invasive procedures with sub-millimeter precision, reducing patient recovery times significantly. Collaborative robots in manufacturing now work safely alongside humans using advanced computer vision and force sensing,

--- Result 3 [biotechnology] ---
at industrial scale. Bioreactor technologies have dramatically reduced the cost of producing monoclonal antibodies, making treatments for autoimmune diseases and cancers more accessible. Microbiome research is revealing how manipulating gut bacteria can influence everything

In [8]:
# MultiQueryRetriever generates multiple alternative phrasings of the user question,
# retrieves docs for each, and returns the deduplicated union — expanding recall
# without requiring the user to manually craft multiple queries

retriever = MultiQueryRetriever.from_llm(retriever=base_retriever, llm=llm)

In [9]:
query = "How are modern technologies improving human health?"

results = retriever.invoke(query)

print(f"Retrieved {len(results)} unique documents:\n")
for i, doc in enumerate(results):
    print(f"--- Result {i+1} [{doc.metadata.get('topic')}] ---")
    print(doc.page_content)
    print()

Retrieved 4 unique documents:

--- Result 1 [genetic_engineering] ---
combined with germline analysis is enabling predictive medicine that identifies disease susceptibility decades before symptoms appear.

--- Result 2 [biotechnology] ---
Biotechnology companies are developing novel protein-based therapies that target specific disease pathways with unprecedented precision. Synthetic biology techniques allow scientists to engineer microorganisms that produce pharmaceutical compounds at industrial scale. Bioreactor technologies have

--- Result 3 [robotics] ---
Surgical robots equipped with haptic feedback allow surgeons to perform minimally invasive procedures with sub-millimeter precision, reducing patient recovery times significantly. Collaborative robots in manufacturing now work safely alongside humans using advanced computer vision and force sensing,

--- Result 4 [genetic_engineering] ---
functional cures for hemophilia B and spinal muscular atrophy. Epigenome editing tools allow 

In [10]:
# include_original=True adds the original user query to the set of queries,
# ensuring docs that match the original phrasing are never missed

retriever_with_original = MultiQueryRetriever.from_llm(
    retriever=base_retriever, 
    llm=llm, 
    include_original=True
)

results_with_original = retriever_with_original.invoke(query)

print(f"Retrieved {len(results_with_original)} unique documents (include_original=True):\n")
for i, doc in enumerate(results_with_original):
    print(f"--- Result {i+1} [{doc.metadata.get('topic')}] ---")
    print(doc.page_content)
    print()

Retrieved 7 unique documents (include_original=True):

--- Result 1 [biotechnology] ---
Biotechnology companies are developing novel protein-based therapies that target specific disease pathways with unprecedented precision. Synthetic biology techniques allow scientists to engineer microorganisms that produce pharmaceutical compounds at industrial scale. Bioreactor technologies have

--- Result 2 [genetic_engineering] ---
combined with germline analysis is enabling predictive medicine that identifies disease susceptibility decades before symptoms appear.

--- Result 3 [genetic_engineering] ---
Base editing and prime editing technologies offer more precise alternatives to CRISPR-Cas9, enabling single-letter corrections to the genome without creating double-strand breaks. Gene therapy trials using adeno-associated virus vectors have achieved functional cures for hemophilia B and spinal

--- Result 4 [robotics] ---
Surgical robots equipped with haptic feedback allow surgeons to perform mi